# GenAI tools in a data science project

Data Scientist track, lesson DS14. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** grade an assistant's modeling answers with the five checks of the core and log accept, revise or reject; grade a text classifier against labeled notes with the metrics you know; report what a run and a classification pass cost in tokens; and say when a classical model is the better tool.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules. The assistant answers and the classifier labels of this lesson are illustrations written for it, not output of any model.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")


con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people, 177,379 events, and 692 people with a cancel_subscription event.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)
CANCELERS = con.execute("SELECT COUNT(DISTINCT user_id) FROM events WHERE event_name = 'cancel_subscription'").fetchall()[0][0]
assert CANCELERS == 692


# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")


## The model and the split

The split of lessons DS2 to DS5 and the day-7 logistic regression of DS4. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)


def logit():
    """The day-7 logistic regression of lessons DS2 to DS5."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


model = logit().fit(X_train, y_train)
print(len(X_train), len(X_test))

## The notes and the labels

60 cancellation notes written for this lesson, and two columns of labels written for it too: an illustration of a text classifier, not a real one. Run this cell as it is.

In [ ]:
LABELS = ["too_expensive", "not_using", "missing_feature", "found_alternative", "other"]      # the five cancel_reason values of app-events
# 60 cancellation notes WRITTEN FOR THIS LESSON (the dataset has no free text), each with the reason it was written to show. The last column marks 5 notes that give two reasons.
NOTES = [
    ("At $7.99 a month it costs more than the savings it finds me.", "too_expensive", 0),
    ("Too expensive for a student budget. Canceling.", "too_expensive", 0),
    ("I can't justify paying every month for this.", "too_expensive", 0),
    ("The annual price is steep; I'm dropping it.", "too_expensive", 0),
    ("Cost is the only reason. The app is fine.", "too_expensive", 0),
    ("Money is tight this month, so subscriptions are going.", "too_expensive", 0),
    ("Paying $60 a year for expense tracking is too much.", "too_expensive", 0),
    ("I'd stay if it were half the price.", "too_expensive", 0),
    ("Can't afford it right now.", "too_expensive", 0),
    ("Pricing is out of line with what I get.", "too_expensive", 0),
    ("The renewal charge surprised me and it's more than I want to pay.", "too_expensive", 0),
    ("Too costly for what is basically a list of expenses.", "too_expensive", 0),
    ("My budget has no room for another monthly fee.", "too_expensive", 0),
    ("I barely use it, but the real reason is the price.", "too_expensive", 1),
    ("I stopped opening it weeks ago.", "not_using", 0),
    ("Haven't used it since January.", "not_using", 0),
    ("I forgot I even had it.", "not_using", 0),
    ("Opened it twice last month, so no point paying.", "not_using", 0),
    ("I don't track expenses anymore.", "not_using", 0),
    ("Not using it, canceling.", "not_using", 0),
    ("I only used it for the first week.", "not_using", 0),
    ("Life got busy and I never come back to it.", "not_using", 0),
    ("I rarely log anything now.", "not_using", 0),
    ("No longer need it.", "not_using", 0),
    ("I set it up and never made it a habit.", "not_using", 0),
    ("I haven't added an expense in a month.", "not_using", 0),
    ("Stopped tracking my spending, so I don't need the app.", "not_using", 0),
    ("I never really got into it.", "not_using", 0),
    ("I don't remember the last time I logged in.", "not_using", 0),
    ("Don't use it enough to keep paying.", "not_using", 0),
    ("I got what I needed in the first month.", "not_using", 0),
    ("Not using it. Please cancel.", "not_using", 0),
    ("My spending is simple now and I don't track it.", "not_using", 0),
    ("The reminders are all I see; I never open the app.", "not_using", 0),
    ("It costs a bit, but mostly I just stopped opening it.", "not_using", 1),
    ("No way to split an expense between two people.", "missing_feature", 0),
    ("It can't import my bank statement.", "missing_feature", 0),
    ("I need shared budgets for my family and there are none.", "missing_feature", 0),
    ("There is no dark mode and no widgets.", "missing_feature", 0),
    ("Missing multi-currency support, which I need for travel.", "missing_feature", 0),
    ("Can't export my data to a spreadsheet.", "missing_feature", 0),
    ("No recurring expense reminders.", "missing_feature", 0),
    ("I need yearly reports and the app only shows months.", "missing_feature", 0),
    ("It lacks categories I can rename.", "missing_feature", 0),
    ("Mainly the missing receipt scanning; the price is also high.", "missing_feature", 1),
    ("Switched to another budgeting app that my bank offers.", "found_alternative", 0),
    ("I moved to a spreadsheet that does what I need.", "found_alternative", 0),
    ("My bank's app now tracks my spending for free.", "found_alternative", 0),
    ("A friend showed me a different app and I like it better.", "found_alternative", 0),
    ("Using a competitor now; it has a cleaner layout.", "found_alternative", 0),
    ("I'm using a notes app to track expenses instead.", "found_alternative", 0),
    ("Found a free app with the same features.", "found_alternative", 0),
    ("Went back to my old spreadsheet.", "found_alternative", 0),
    ("Another app is cheaper and syncs with my bank.", "found_alternative", 1),
    ("Closing my account because I'm moving abroad.", "other", 0),
    ("Please delete everything; I'm worried about my privacy.", "other", 0),
    ("I'm taking a break from finances for a while.", "other", 0),
    ("Billing mistake: I was charged twice.", "other", 0),
    ("Mostly the crash on my phone; I also stopped using it.", "other", 1),
    ("Canceling for personal reasons.", "other", 0),
]
notes = pd.DataFrame(NOTES, columns=["text", "truth", "hard"])
# The labels of an "LLM classifier", ALSO WRITTEN FOR THIS LESSON: an illustration of the kinds of errors a grader meets, not the output of any model.
# Run 1 and run 2 are two runs of the same prompt. Their accuracy says nothing about any real model.
RUN1 = {"It costs a bit, but mostly I just stopped opening it.": "too_expensive", "Another app is cheaper and syncs with my bank.": "too_expensive",
        "I barely use it, but the real reason is the price.": "not_using", "Mostly the crash on my phone; I also stopped using it.": "not_using",
        "Canceling for personal reasons.": "not_using", "Billing mistake: I was charged twice.": "too_expensive", "It lacks categories I can rename.": "other",
        "Can't export my data to a spreadsheet.": "found_alternative", "I'm using a notes app to track expenses instead.": "other", "I forgot I even had it.": "forgotten"}
RUN2 = {"Another app is cheaper and syncs with my bank.": "too_expensive", "I barely use it, but the real reason is the price.": "not_using",
        "Mostly the crash on my phone; I also stopped using it.": "not_using", "Billing mistake: I was charged twice.": "too_expensive", "It lacks categories I can rename.": "other",
        "Can't export my data to a spreadsheet.": "found_alternative", "I'm using a notes app to track expenses instead.": "other", "I'd stay if it were half the price.": "other",
        "No longer need it.": "other", "I got what I needed in the first month.": "found_alternative"}
notes["run1"] = [RUN1.get(t, truth) for t, truth in zip(notes["text"], notes["truth"])]
notes["run2"] = [RUN2.get(t, truth) for t, truth in zip(notes["text"], notes["truth"])]
print(len(notes), notes["truth"].value_counts().to_dict())

## Unit 1: Grade the assistant's answers

**Predict first.** Four answers came back from an assistant, each with a claim. Which do you accept, which do you revise, and do you reject any?

In [ ]:
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", "passthrough", WEEK1)])
Xe_train, Xe_test = prep.fit_transform(X_train), prep.transform(X_test)
deep = DecisionTreeClassifier(random_state=SEED).fit(Xe_train, y_train)                  # the assistant's tree, with no limit on its depth
paid = table["channel"] == "paid_social"
a = (len(X_train), len(X_test), round(float(y_train.mean()), 3), round(float(y_test.mean()), 3))                          # answer A: the split
b = (round(float(roc_auc_score(y_test, model.predict(X_test))), 3), round(float(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])), 3))      # answer B: hard labels, then chances
c = (round(float(deep.score(Xe_train, y_train)), 3), round(float(deep.score(Xe_test, y_test)), 3))                          # answer C: training rows, then test rows
d = (round(float(table.loc[paid, "churned"].mean()), 3), round(float(table.loc[~paid, "churned"].mean()), 3), round(float(table["churned"].mean()), 3))      # answer D
log = pd.DataFrame([
    ("A split", f"counted {a[0]:,} and {a[1]:,}; churners {a[2]} and {a[3]}", "accept", "The counts and the shares match the claim."),
    ("B ROC AUC", f"with chances {b[1]}, with hard labels {b[0]}", "revise", "ROC AUC needs the chances: predict_proba."),
    ("C tree", f"training rows {c[0]}, test rows {c[1]}", "revise", "The score was read on the rows the tree learned from."),
    ("D cut paid social", f"churn {d[0]} for paid social, {d[1]} for the rest, {d[2]} for all", "reject", "A gap between groups is not what a change would do."),
], columns=["answer", "check", "verdict", "reason"])
print(log.to_string(index=False))

## Unit 2: Grade a classifier against labels

**Predict first.** Run 1 labels 60 notes. About how many of its answers are right: about 0.55, about 0.85 or about 0.98? And how many are not one of the five labels?

In [ ]:
valid = notes["run1"].isin(LABELS)
pred = notes["run1"].where(valid, "invalid")                                      # an answer outside the five labels counts as wrong
hit = (pred == notes["truth"]).to_numpy()
print(int((~valid).sum()), round(float(hit.mean()), 3), round(float(notes["truth"].value_counts(normalize=True).max()), 3))
print(pd.crosstab(notes["truth"], pred).reindex(index=LABELS, columns=LABELS + ["invalid"], fill_value=0).to_string())
recall = pd.Series({k: float(hit[(notes["truth"] == k).to_numpy()].mean()) for k in LABELS})
precision = pd.Series({k: float(hit[(pred == k).to_numpy()].mean()) for k in LABELS})
print(pd.DataFrame({"recall": recall.round(2), "precision": precision.round(2)}).T.to_string())
rng = np.random.default_rng(SEED)
accs = [hit[rng.integers(0, len(hit), len(hit))].mean() for _ in range(2000)]      # resample the 60 notes
print(np.round(np.percentile(accs, [2.5, 97.5]), 3))

## Unit 3: Two runs, and what a pass costs

**Predict first.** Classifying the notes of the 692 people who canceled with the small model: will the bill be under a cent, about a dollar, or about a hundred dollars?

In [ ]:
agree = float((notes["run1"] == notes["run2"]).mean())
print(round(agree, 3), int((notes["run1"] != notes["run2"]).sum()))
PROMPT = "Classify this cancellation note as one of: too_expensive, not_using, missing_feature, found_alternative, other. Answer with the label only.\nNote: "
PRICE = {"Haiku 5.5": (0.10, 0.50), "Sonnet 5.5": (2.00, 10.00)}               # US dollars per million input and output tokens, from the vendor's pricing page of 2026-11-24 (Haiku 5.5: for prompts up to 100,000 tokens)
CHARS_PER_TOKEN = 4                                                              # a rule of thumb for English text, not the count of any tokenizer
tokens_in = (len(PROMPT) + notes["text"].str.len()) / CHARS_PER_TOKEN
tokens_out = notes["run1"].str.len() / CHARS_PER_TOKEN
per_note = (float(tokens_in.mean()), float(tokens_out.mean()))
print(round(per_note[0], 1), round(per_note[1], 1))
ANSWERS = ["A. The split: 13,362 to fit, 4,454 to test, 54.4% churners in both parts.", "B. The logistic regression has a ROC AUC of 0.62 on the test people.",
           "C. An unlimited decision tree scores 0.791 accuracy.", "D. Paid-social people churn 3.2 times as often, so stopping paid social would cut churn by two thirds."]
context = MODELING_SQL + table.head(5).to_csv()                                  # what the assistant was given: the table's definition and five rows
run_in, run_out = len(context) / CHARS_PER_TOKEN, len(" ".join(ANSWERS)) / CHARS_PER_TOKEN
print(round(run_in), round(run_out), f"{(run_in * PRICE['Sonnet 5.5'][0] + run_out * PRICE['Sonnet 5.5'][1]) / 1e6:.4f}")      # the floor of one run: no code, no retries
for model_name, (p_in, p_out) in PRICE.items():
    cost = (per_note[0] * p_in + per_note[1] * p_out) / 1e6                       # dollars for one note
    print(model_name, f"{cost * CANCELERS:.4f}", f"{cost * 1_000_000:,.2f}")       # the 692 people who canceled, and a million notes

## Unit 4: Or a model you train

**Predict first.** With 60 labeled notes, will a classical text model trained on 48 of them score about 0.95, about 0.80 or under 0.6 on the other 12?

In [ ]:
text_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
for folds in (3, 5, 6):
    scores = cross_val_score(text_model, notes["text"], notes["truth"], cv=StratifiedKFold(folds, shuffle=True, random_state=SEED))
    print(folds, round(60 - 60 / folds), round(float(scores.mean()), 3), round(float(scores.min()), 3), round(float(scores.max()), 3))

## Finish it

### Exercise 1: Finish it

**Task:** **Recall by label, run 2.** Unit 2 graded run 1. Grade run 2 the same way: for each label, the share of the notes with that true label that run 2 got right, counting an answer outside the five labels as wrong. Put the five recalls, in the order of `LABELS`, in `answer_1`, each rounded to 3 decimals.

**Expected:** `answer_1` is a list of five numbers.

In [ ]:
pred2 = notes["run2"].where(notes["run2"].isin(LABELS), "invalid")
hit2 = (pred2 == notes["truth"]).to_numpy()
# recall of run 2 for each label: the share of the notes with that true label that run 2 got right
answer_1 = None  # replace None with the five recalls in the order of LABELS, each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [0.857, 0.905, 0.8, 0.778, 0.667], tol=2e-3)

<details>
<summary>Hint</summary>

`hit2` says whether each note was right. For each label, keep the notes whose `truth` is that label and take the mean of `hit2` over them.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [round(float(hit2[(notes["truth"] == k).to_numpy()].mean()), 3) for k in ...]
```

</details>

<details>
<summary>Solution</summary>

```python
pred2 = notes["run2"].where(notes["run2"].isin(LABELS), "invalid")
hit2 = (pred2 == notes["truth"]).to_numpy()
answer_1 = [round(float(hit2[(notes["truth"] == k).to_numpy()].mean()), 3) for k in LABELS]
```

Run 2 gets 0.857 of the too-expensive notes, 0.905 of the not-using notes, 0.8 of the missing-feature notes, 0.778 of the found-alternative notes and 0.667 of the other notes. Its overall accuracy is the same as run 1's, but the notes it gets wrong are not the same ones: an accuracy hides which notes fail, and recall by label shows it.

</details>

## On your own

### Exercise 2: On your own

**Task:** **A longer prompt.** Before you start, say in one sentence what you expect: a prompt with worked examples costs more for every note, and the answer is still cents. Now the instructions are 640 characters long (three worked examples), the model is Sonnet 5.5, and the answer is the label as before. Use `per_note`, `PRICE` and `CHARS_PER_TOKEN` of unit 3 and put the list [input tokens for each note, dollars for the 692 people who canceled, rounded to 3 decimals] in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers, tokens rounded to 1 decimal.

In [ ]:
# write your code here

In [ ]:
_ = check("Exercise 2", "answer_2", [170.6, 0.256], traps=[([47.1, 0.085], "That is the short prompt of unit 3. Use 640 characters of instructions and Sonnet 5.5.")], tol=2e-3)

<details>
<summary>Hint</summary>

Input tokens for a note are the characters of the prompt plus the characters of the note, divided by `CHARS_PER_TOKEN`; use the mean note length. The output tokens are `per_note[1]`.

</details>

<details>
<summary>Half-solution</summary>

```python
tokens_in2 = (640 + notes["text"].str.len().mean()) / CHARS_PER_TOKEN
cost_each = (tokens_in2 * ... + per_note[1] * ...) / 1e6
answer_2 = [round(float(tokens_in2), 1), round(float(cost_each * CANCELERS), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
LONG_PROMPT_CHARS = 640                    # the same task with three worked examples in the prompt
tokens_in2 = (LONG_PROMPT_CHARS + notes["text"].str.len().mean()) / CHARS_PER_TOKEN
p_in, p_out = PRICE["Sonnet 5.5"]
cost_each = (tokens_in2 * p_in + per_note[1] * p_out) / 1e6
answer_2 = [round(float(tokens_in2), 1), round(float(cost_each * CANCELERS), 3)]
```

A prompt of 640 characters gives 170.6 input tokens for each note, about 3.6 times the 47.1 of the short prompt, and the bill for the 692 people who canceled, on Sonnet 5.5 is $0.256, three times the $0.085 of unit 3. The prompt dominates the input, so a longer prompt, or examples, is where the cost grows. These are estimates from 4 characters a token; the usage an API returns is the bill.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague grades run 1 on the 55 notes that give one reason and leaves out the 5 that give two, "because those are ambiguous anyway", and reports the accuracy of the starter. Put in `answer_3` the list [accuracy of run 1 on the 55 one-reason notes, accuracy of run 1 on the 5 two-reason notes], each rounded to 3 decimals.

**Expected:** `answer_3` is a list of two numbers.

In [ ]:
easy = notes["hard"] == 0
colleague = round(float((notes.loc[easy, "run1"] == notes.loc[easy, "truth"]).mean()), 3)      # colleague: "the five two-reason notes are ambiguous anyway, so I left them out"
answer_3 = None  # replace None with [accuracy of run 1 on the 55 one-reason notes, accuracy of run 1 on the 5 two-reason notes], each rounded to 3 decimals

In [ ]:
_ = check("Exercise 3", "answer_3", [0.891, 0.2], traps=[([0.891, 0.891], "The second number is the accuracy on the two-reason notes only.")], tol=2e-3)

<details>
<summary>Hint</summary>

`notes["hard"]` is 1 for the two-reason notes. Take the mean of `run1 == truth` inside each group.

</details>

<details>
<summary>Half-solution</summary>

```python
easy = notes["hard"] == 0
answer_3 = [round(float((notes.loc[easy, "run1"] == notes.loc[easy, "truth"]).mean()), 3), ...]
```

</details>

<details>
<summary>Solution</summary>

```python
easy = notes["hard"] == 0
answer_3 = [round(float((notes.loc[easy, "run1"] == notes.loc[easy, "truth"]).mean()), 3), round(float((notes.loc[~easy, "run1"] == notes.loc[~easy, "truth"]).mean()), 3)]
```

Run 1 is right on 0.891 of the 55 one-reason notes and on 0.2 of the 5 two-reason notes: one right out of five. Leaving the hard notes out raised the accuracy from 0.833 to 0.891 and hid the notes where a real reader would also disagree. With 5 notes the 0.2 is itself very uncertain; the point is that the hard notes need their own line in the grade.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")